In [1]:
import pandas as pd
import numpy as np
import os
from datetime import datetime, timedelta
import random

np.random.seed(42)
random.seed(42)

NUM_USERS       = 500
NUM_DAYS        = 90
MALICIOUS_USERS = 15
START_DATE      = datetime(2023, 1, 1)
OUTPUT_DIR      = "data"

os.makedirs(OUTPUT_DIR, exist_ok=True)

all_users = [f"U{str(i).zfill(4)}" for i in range(1, NUM_USERS + 1)]
bad_users = random.sample(all_users, MALICIOUS_USERS)

print(f"Total users     : {NUM_USERS}")
print(f"Malicious users : {len(bad_users)}")
print("Setup complete!")

Total users     : 500
Malicious users : 15
Setup complete!


In [2]:
def rand_time(date, after_hours=False):
    if after_hours:
        hour = random.choice(list(range(19, 24)) + list(range(0, 6)))
    else:
        hour = random.randint(7, 18)
    minute = random.randint(0, 59)
    second = random.randint(0, 59)
    return date + timedelta(hours=hour, minutes=minute, seconds=second)

print("Helper function ready!")

Helper function ready!


In [3]:
print("Generating logon data... please wait...")
logon_rows = []
pcs = [f"PC{str(i).zfill(4)}" for i in range(1, 201)]

for day_offset in range(NUM_DAYS):
    current_date = START_DATE + timedelta(days=day_offset)
    is_weekend = current_date.weekday() >= 5

    for user in all_users:
        is_bad = user in bad_users
        if is_weekend:
            login_prob = 0.60 if is_bad else 0.10
        else:
            login_prob = 0.95
        if random.random() > login_prob:
            continue
        num_logins = random.randint(2, 6) if is_bad and is_weekend else random.randint(1, 3)
        for _ in range(num_logins):
            ts = rand_time(current_date, after_hours=(is_bad and random.random() < 0.4))
            pc = random.choice(pcs)
            if is_bad and random.random() < 0.3:
                pc = f"PC{str(random.randint(180, 200)).zfill(4)}"
            logon_rows.append({
                "id": f"L{len(logon_rows):07d}",
                "date": ts.strftime("%m/%d/%Y %H:%M:%S"),
                "user": user,
                "pc": pc,
                "activity": "Logon"
            })

logon_df = pd.DataFrame(logon_rows)
logon_df.to_csv(f"{OUTPUT_DIR}/logon.csv", index=False)
print(f"Done! {len(logon_df):,} logon events saved!")
logon_df.head()

Generating logon data... please wait...
Done! 65,400 logon events saved!


,id,date,user,pc,activity
0,L0000000,01/01/2023 10:32:38,U0002,PC0007,Logon
1,L0000001,01/01/2023 12:06:05,U0013,PC0191,Logon
2,L0000002,01/01/2023 07:46:29,U0013,PC0192,Logon
3,L0000003,01/01/2023 23:53:40,U0013,PC0159,Logon
4,L0000004,01/01/2023 10:49:18,U0016,PC0021,Logon


In [5]:
print("Generating file access data... please wait...")
file_rows = []
file_types      = ["docx", "xlsx", "pdf", "txt", "pptx", "zip", "csv"]
sensitive_types = ["zip", "csv", "pdf"]

for day_offset in range(NUM_DAYS):
    current_date = START_DATE + timedelta(days=day_offset)
    for user in all_users:
        is_bad = user in bad_users
        threat_ramp = min(1.0, day_offset / NUM_DAYS)
        if is_bad:
            num_files = int(random.randint(3, 8) + threat_ramp * random.randint(10, 35))
        else:
            num_files = random.randint(0, 8)
        for _ in range(num_files):
            ts = rand_time(current_date, after_hours=(is_bad and random.random() < 0.35))
            ftype = random.choice(sensitive_types if is_bad and random.random() < 0.5 else file_types)
            file_rows.append({
                "id": f"F{len(file_rows):07d}",
                "date": ts.strftime("%m/%d/%Y %H:%M:%S"),
                "user": user,
                "filename": f"file_{random.randint(1000,9999)}.{ftype}",
                "activity": random.choice(["open", "copy", "write", "delete"])
            })

file_df = pd.DataFrame(file_rows)
file_df.to_csv(f"{OUTPUT_DIR}/file.csv", index=False)
print(f"Done! {len(file_df):,} file events saved!")
file_df.head()

Generating file access data... please wait...
Done! 197,387 file events saved!


,id,date,user,filename,activity
0,F0000000,01/01/2023 08:36:07,U0001,file_8182.docx,copy
1,F0000001,01/01/2023 11:39:23,U0001,file_5266.zip,open
2,F0000002,01/01/2023 14:51:37,U0002,file_9549.txt,copy
3,F0000003,01/01/2023 12:50:04,U0002,file_5551.xlsx,write
4,F0000004,01/01/2023 17:49:31,U0002,file_7001.zip,open


In [6]:
print("Generating email data... please wait...")
email_rows = []
external_domains = ["gmail.com", "yahoo.com", "hotmail.com", "protonmail.com"]
internal_domain  = "company.com"

for day_offset in range(NUM_DAYS):
    current_date = START_DATE + timedelta(days=day_offset)
    for user in all_users:
        is_bad = user in bad_users
        num_emails = random.randint(5, 20) if is_bad else random.randint(2, 10)
        for _ in range(num_emails):
            ts = rand_time(current_date, after_hours=(is_bad and random.random() < 0.3))
            if is_bad and random.random() < 0.45:
                to_addr = f"external_{random.randint(1,50)}@{random.choice(external_domains)}"
            else:
                to_addr = f"user_{random.randint(1,500)}@{internal_domain}"
            has_attachment = random.random() < (0.6 if is_bad else 0.2)
            email_rows.append({
                "id": f"E{len(email_rows):07d}",
                "date": ts.strftime("%m/%d/%Y %H:%M:%S"),
                "user": user,
                "to": to_addr,
                "cc": "",
                "bcc": "",
                "size": random.randint(1000, 50000) if not has_attachment else random.randint(50000, 5000000),
                "attachments": random.randint(1, 5) if has_attachment else 0,
                "activity": "Send"
            })

email_df = pd.DataFrame(email_rows)
email_df.to_csv(f"{OUTPUT_DIR}/email.csv", index=False)
print(f"Done! {len(email_df):,} email events saved!")
email_df.head()

Generating email data... please wait...
Done! 277,745 email events saved!


,id,date,user,to,cc,bcc,size,attachments,activity
0,E0000000,01/01/2023 11:48:58,U0001,user_120@company.com,,,9098,0,Send
1,E0000001,01/01/2023 13:36:13,U0001,user_485@company.com,,,14590,0,Send
2,E0000002,01/01/2023 10:44:42,U0001,user_115@company.com,,,22413,0,Send
3,E0000003,01/01/2023 12:00:21,U0001,user_341@company.com,,,15629,0,Send
4,E0000004,01/01/2023 11:30:29,U0001,user_108@company.com,,,8058,0,Send


In [7]:
print("Generating device/USB data... please wait...")
device_rows = []

for day_offset in range(NUM_DAYS):
    current_date = START_DATE + timedelta(days=day_offset)
    for user in all_users:
        is_bad = user in bad_users
        use_usb = random.random() < (0.35 if is_bad else 0.03)
        if not use_usb:
            continue
        ts = rand_time(current_date, after_hours=(is_bad and random.random() < 0.5))
        device_rows.append({
            "id": f"D{len(device_rows):07d}",
            "date": ts.strftime("%m/%d/%Y %H:%M:%S"),
            "user": user,
            "pc": f"PC{str(random.randint(1,200)).zfill(4)}",
            "activity": "Connect"
        })

device_df = pd.DataFrame(device_rows)
device_df.to_csv(f"{OUTPUT_DIR}/device.csv", index=False)
print(f"Done! {len(device_df):,} device events saved!")
device_df.head()

Generating device/USB data... please wait...
Done! 1,826 device events saved!


,id,date,user,pc,activity
0,D0000000,01/01/2023 04:23:20,U0013,PC0198,Connect
1,D0000001,01/01/2023 14:34:39,U0021,PC0092,Connect
2,D0000002,01/01/2023 09:30:43,U0045,PC0104,Connect
3,D0000003,01/01/2023 10:35:08,U0050,PC0059,Connect
4,D0000004,01/01/2023 09:07:28,U0075,PC0101,Connect


In [9]:
total_events = len(logon_df) + len(file_df) + len(email_df) + len(device_df)

print("=" * 50)
print("  DATA GENERATION COMPLETE")
print("=" * 50)
print(f"  Total events    : {total_events:,}")
print(f"  Logon events    : {len(logon_df):,}")
print(f"  File events     : {len(file_df):,}")
print(f"  Email events    : {len(email_df):,}")
print(f"  Device events   : {len(device_df):,}")
print(f"  Insider users   : {len(bad_users)}")
print("=" * 50)
print("Data generation complete!")

  DATA GENERATION COMPLETE
  Total events    : 542,310
  Logon events    : 65,400
  File events     : 197,387
  Email events    : 277,745
  Device events   : 1,778
  Insider users   : 15
Data generation complete!
